# Lab | LangChain Evaluation — VS Code versie

**Wat doet dit notebook?**
Dit notebook bouwt een RAG-systeem (Retrieval-Augmented Generation) en evalueert het op drie manieren:
1. **Handmatige evaluatie** — je kijkt zelf naar de output via debug mode
2. **LLM-as-a-Judge** — een LLM scoort de output
3. **RAGAS evaluatie** — gespecialiseerde metrics voor RAG-systemen

---
### Vereisten (eenmalige setup)

**1. Maak een virtual environment aan in je terminal:**
```bash
python -m venv venv
venv\\Scripts\\activate        # Windows
source venv/bin/activate      # Mac/Linux
```

**2. Installeer packages:**
```bash
pip install langchain==0.2.16 langchain-community==0.2.16 langchain-openai langchain-huggingface docarray ragas datasets python-dotenv
```

**3. Maak een `.env` bestand aan** (zie `.env.example`) met je OpenAI API key.

**4. Zet de databestanden** in dezelfde map als dit notebook:
- `OutdoorClothingCatalog_1000(2).csv`
- `nyc_text.txt`

**5. Selecteer je venv als kernel** in VS Code (rechts boven in het notebook: klik op de kernel selector).

## Stap 0 — Imports & API Key laden

In [1]:
import os
from dotenv import load_dotenv

# Laad de .env file uit dezelfde map als dit notebook
load_dotenv()

OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')

if not OPENAI_API_KEY:
    raise ValueError('❌ OPENAI_API_KEY niet gevonden! Maak een .env bestand aan met OPENAI_API_KEY=sk-...')

os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY
print('✓ API key geladen')

✓ API key geladen


In [2]:
from langchain.chains import RetrievalQA, LLMChain
from langchain.indexes import VectorstoreIndexCreator
from langchain_openai import ChatOpenAI
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import CSVLoader, TextLoader
from langchain_community.vectorstores import DocArrayInMemorySearch
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
print('✓ Imports klaar')

✓ Imports klaar


---
# EXAMPLE 1 — Outdoor Clothing Catalog (CSV)

Zorg dat `OutdoorClothingCatalog_1000(2).csv` in dezelfde map staat als dit notebook.

## 1A — Bouw het RAG-systeem

In [4]:
# Pad relatief aan dit notebook — geen /content/ meer!
file = '/Users/domiendarmont/Desktop/Ironhack/lab_week_8/lab-langchain-evaluation/data/OutdoorClothingCatalog_1000.csv'
loader = CSVLoader(file_path=file)
data = loader.load()
print(f'✓ {len(data)} documenten geladen uit CSV')
print('Voorbeeld document:', data[0].page_content[:200])

✓ 1000 documenten geladen uit CSV
Voorbeeld document: : 0
name: Women's Campside Oxfords
description: This ultracomfortable lace-to-toe Oxford boasts a super-soft canvas, thick cushioning, and quality construction for a broken-in feel from the first time


In [5]:
# HuggingFace embeddings — gratis, geen API key nodig
# Altijd 'cpu' lokaal, tenzij je een NVIDIA GPU hebt met CUDA drivers
embeddings = HuggingFaceEmbeddings(
    model_name='all-MiniLM-L6-v2',
    model_kwargs={'device': 'cpu'}
)

index = VectorstoreIndexCreator(
    vectorstore_cls=DocArrayInMemorySearch,
    embedding=embeddings
).from_loaders([loader])

vectorstore = index.vectorstore
retriever = vectorstore.as_retriever()
print('✓ Vectorstore aangemaakt')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✓ Vectorstore aangemaakt


In [6]:
llm = ChatOpenAI(temperature=0.0, openai_api_key=OPENAI_API_KEY)

qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type='stuff',
    retriever=retriever,
    return_source_documents=True,
    verbose=True,
    chain_type_kwargs={
        'document_separator': '<<<<>>>>>'
    }
)
print('✓ QA chain klaar')

✓ QA chain klaar


## 1B — Bekijk de data

In [7]:
print('Document 10:')
print(data[10].page_content)
print('\nDocument 11:')
print(data[11].page_content)

Document 10:
: 10
name: Cozy Comfort Pullover Set, Stripe
description: Perfect for lounging, this striped knit set lives up to its name. We used ultrasoft fabric and an easy design that's as comfortable at bedtime as it is when we have to make a quick run out.

Size & Fit
- Pants are Favorite Fit: Sits lower on the waist.
- Relaxed Fit: Our most generous fit sits farthest from the body.

Fabric & Care
- In the softest blend of 63% polyester, 35% rayon and 2% spandex.

Additional Features
- Relaxed fit top with raglan sleeves and rounded hem.
- Pull-on pants have a wide elastic waistband and drawstring, side pockets and a modern slim leg.

Imported.

Document 11:
: 11
name: Ultra-Lofty 850 Stretch Down Hooded Jacket
description: This technical stretch down jacket from our DownTek collection is sure to keep you warm and comfortable with its full-stretch construction providing exceptional range of motion. With a slightly fitted style that falls at the hip and best with a midweight layer, 

## 1C — Handgemaakte testvoorbeelden

In [8]:
examples = [
    {
        'query': 'Do the Cozy Comfort Pullover Set have side pockets?',
        'answer': 'Yes'
    },
    {
        'query': 'What collection is the Ultra-Lofty 850 Stretch Down Hooded Jacket from?',
        'answer': 'The DownTek collection'
    }
]
print(f'✓ {len(examples)} handgemaakte voorbeelden')

✓ 2 handgemaakte voorbeelden


## 1D — LLM-gegenereerde testvoorbeelden

Een LLM genereert automatisch Q&A-paren uit de documenten — handig als je geen tijd hebt om zelf testdata te maken.

In [11]:
# Skip LLM-generatie (deprecated + buggy in deze versie)
print(f'✓ Doorgaan met {len(examples)} handgemaakte voorbeelden')



✓ Doorgaan met 2 handgemaakte voorbeelden


In [13]:
print(f'✓ Totaal {len(examples)} voorbeelden')
print('Eerste voorbeeld:', examples[0])

✓ Totaal 2 voorbeelden
Eerste voorbeeld: {'query': 'Do the Cozy Comfort Pullover Set have side pockets?', 'answer': 'Yes'}


## 1E — Handmatige evaluatie (debug mode)

Met `langchain.debug = True` zie je exact welke documenten worden opgehaald en welke prompt naar de LLM gaat.

In [14]:
import langchain

langchain.debug = True

result = qa.invoke({'query': examples[0]['query']})
print('\n=== ANTWOORD ===')
print(result['result'])

langchain.debug = False

[chain/start] [chain:RetrievalQA] Entering Chain run with input:
{
  "query": "Do the Cozy Comfort Pullover Set have side pockets?"
}
[chain/start] [chain:RetrievalQA > chain:StuffDocumentsChain] Entering Chain run with input:
[inputs]
[chain/start] [chain:RetrievalQA > chain:StuffDocumentsChain > chain:LLMChain] Entering Chain run with input:
{
  "question": "Do the Cozy Comfort Pullover Set have side pockets?",
  "context": ": 73\nname: Cozy Cuddles Knit Pullover Set\ndescription: Perfect for lounging, this knit set lives up to its name. We used ultrasoft fabric and an easy design that's as comfortable at bedtime as it is when we have to make a quick run out. \n\nSize & Fit \nPants are Favorite Fit: Sits lower on the waist. \nRelaxed Fit: Our most generous fit sits farthest from the body. \n\nFabric & Care \nIn the softest blend of 63% polyester, 35% rayon and 2% spandex.\n\nAdditional Features \nRelaxed fit top with raglan sleeves and rounded hem. \nPull-on pants have a wide elastic

## 1F — LLM-as-a-Judge evaluatie

Een tweede LLM beoordeelt de antwoorden op een schaal van 0 tot 1.

In [15]:
llm_eval = ChatOpenAI(model='gpt-3.5-turbo', temperature=0, openai_api_key=OPENAI_API_KEY)

predictions = []
for eg in examples[:5]:
    result = qa.invoke({'query': eg['query']})
    predictions.append({
        'query': eg['query'],
        'answer': eg['answer'],
        'result': result['result'],
        'source_documents': result.get('source_documents', [])
    })

print(f'✓ {len(predictions)} voorspellingen gegenereerd')
for i, p in enumerate(predictions):
    print(f'\nVoorbeeld {i}:')
    print(f'  Vraag:            {p["query"]}')
    print(f'  Correct antwoord: {p["answer"]}')
    print(f'  Model antwoord:   {p["result"]}')
    print(f'  Bronnen gebruikt: {len(p["source_documents"])}')



> Entering new RetrievalQA chain...

> Finished chain.


> Entering new RetrievalQA chain...

> Finished chain.
✓ 2 voorspellingen gegenereerd

Voorbeeld 0:
  Vraag:            Do the Cozy Comfort Pullover Set have side pockets?
  Correct antwoord: Yes
  Model antwoord:   Yes, the Cozy Comfort Pullover Set does have side pockets.
  Bronnen gebruikt: 4

Voorbeeld 1:
  Vraag:            What collection is the Ultra-Lofty 850 Stretch Down Hooded Jacket from?
  Correct antwoord: The DownTek collection
  Model antwoord:   The Ultra-Lofty 850 Stretch Down Hooded Jacket is from the DownTek collection.
  Bronnen gebruikt: 4


In [16]:
import re

eval_prompt = PromptTemplate(
    input_variables=['query', 'answer', 'result'],
    template="""Geef een score van 0 tot 1 voor hoe goed het VOORSPELDE antwoord overeenkomt met het CORRECTE antwoord voor de vraag.

Vraag: {query}
Correct antwoord: {answer}
Voorspeld antwoord: {result}

Geef ALLEEN een getal tussen 0 en 1. Niets anders.
Score:"""
)

eval_chain = eval_prompt | llm_eval | StrOutputParser()

graded_outputs = []
for i, pred in enumerate(predictions):
    score_raw = eval_chain.invoke({
        'query': pred['query'],
        'answer': pred['answer'],
        'result': pred['result']
    })
    try:
        score = float(score_raw.strip())
    except ValueError:
        match = re.search(r'[0-9.]+', score_raw)
        score = float(match.group()) if match else 0.0

    graded_outputs.append({'score': score})
    print(f'Voorbeeld {i}: score = {score:.2f}  | vraag: "{pred["query"][:60]}"')

avg_score = sum(g['score'] for g in graded_outputs) / len(graded_outputs)
print(f'\n✓ Gemiddelde LLM-Judge score: {avg_score:.3f}')

Voorbeeld 0: score = 1.00  | vraag: "Do the Cozy Comfort Pullover Set have side pockets?"
Voorbeeld 1: score = 1.00  | vraag: "What collection is the Ultra-Lofty 850 Stretch Down Hooded J"

✓ Gemiddelde LLM-Judge score: 1.000


---
# EXAMPLE 2 — NYC Wikipedia Tekst (RAGAS evaluatie)

Zorg dat `nyc_text.txt` in dezelfde map staat als dit notebook.

RAGAS meet:
- **Faithfulness** — Verzint het model dingen, of blijft het bij de bronnen?
- **Context Recall** — Haalt het systeem de juiste documenten op?

## 2A — Bouw het RAG-systeem voor NYC

In [17]:
# Lokaal pad — geen /content/ meer
loader_nyc = TextLoader('/Users/domiendarmont/Desktop/Ironhack/lab_week_8/lab-langchain-evaluation/data/nyc_text.txt')

# Lokaal altijd 'cpu' — tenzij je CUDA hebt
embeddings_nyc = HuggingFaceEmbeddings(
    model_name='all-MiniLM-L6-v2',
    model_kwargs={'device': 'cpu'}
)

index_nyc = VectorstoreIndexCreator(
    embedding=embeddings_nyc
).from_loaders([loader_nyc])

retriever_nyc = index_nyc.vectorstore.as_retriever()

llm_nyc = ChatOpenAI(temperature=0, openai_api_key=OPENAI_API_KEY)

qa_chain = RetrievalQA.from_chain_type(
    llm_nyc,
    retriever=retriever_nyc,
    return_source_documents=True,
)

print('✓ NYC RAG chain klaar')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

/opt/homebrew/lib/python3.11/site-packages/langchain/indexes/vectorstore.py:127: UserWarning: Using InMemoryVectorStore as the default vectorstore.This memory store won't persist data. You should explicitlyspecify a vectorstore when using VectorstoreIndexCreator
  warnings.warn(


✓ NYC RAG chain klaar


In [18]:
# Snelle test
vraag = 'How did New York City get its name?'
test_result = qa_chain.invoke({'query': vraag})
print('Vraag:', vraag)
print('Antwoord:', test_result['result'])

Vraag: How did New York City get its name?
Antwoord: New York City was originally named New Amsterdam by Dutch colonists in 1626. When the city came under British control in 1664, it was renamed New York after King Charles II of England granted the lands to his brother, the Duke of York. The city has been continuously named New York since November 1674.


## 2B — Testset met ground truth

In [19]:
eval_questions = [
    'What is the population of New York City as of 2020?',
    'Which borough of New York City has the highest population?',
    'What is the economic significance of New York City?',
    'How did New York City get its name?',
    'What is the significance of the Statue of Liberty in New York City?',
]

eval_answers = [
    '8,804,190',
    'Brooklyn',
    "New York City's economic significance is vast, as it serves as the global financial capital, housing Wall Street and major financial institutions. Its diverse economy spans technology, media, healthcare, education, and more, making it resilient to economic fluctuations.",
    'New York City got its name when it came under British control in 1664. King Charles II of England granted the lands to his brother, the Duke of York, who named the city New York in his own honor.',
    'The Statue of Liberty in New York City holds great significance as a symbol of the United States and its ideals of liberty and peace. It greeted millions of immigrants who arrived in the U.S. by ship in the late 19th and early 20th centuries.',
]

examples_nyc = [
    {'query': q, 'ground_truths': [eval_answers[i]]}
    for i, q in enumerate(eval_questions)
]

print(f'✓ {len(examples_nyc)} testvragen klaargezet')

✓ 5 testvragen klaargezet


## 2C — Genereer voorspellingen

In [20]:
all_predictions = []
for example in examples_nyc:
    q = example['query']
    result = qa_chain.invoke({'query': q})
    all_predictions.append({
        'question': q,
        'answer': result['result'],
        'contexts': [doc.page_content for doc in result['source_documents']],
        'ground_truth': example['ground_truths'][0]
    })

print(f'✓ {len(all_predictions)} voorspellingen gegenereerd')
print('\nEerste voorspelling:')
print('  Vraag:', all_predictions[0]['question'])
print('  Antwoord:', all_predictions[0]['answer'])
print('  Bronnen:', len(all_predictions[0]['contexts']), 'documenten')
print('  Ground truth:', all_predictions[0]['ground_truth'])

✓ 5 voorspellingen gegenereerd

Eerste voorspelling:
  Vraag: What is the population of New York City as of 2020?
  Antwoord: The population of New York City as of 2020 is 8,804,190 residents.
  Bronnen: 4 documenten
  Ground truth: 8,804,190


## 2D — RAGAS Evaluatie

### Sectie 1: Enkele evaluatie (voor begrip)

In [21]:
from ragas.integrations.langchain import EvaluatorChain
from ragas.metrics import faithfulness, context_recall

faithfulness_chain = EvaluatorChain(metric=faithfulness)
context_recall_chain = EvaluatorChain(metric=context_recall)

print('✓ RAGAS evaluators aangemaakt')

✓ RAGAS evaluators aangemaakt


/opt/homebrew/lib/python3.11/site-packages/pydantic/v1/main.py:1019: RuntimeWarning: fields may not start with an underscore, ignoring "_required_columns"
  warnings.warn(f'fields may not start with an underscore, ignoring "{f_name}"', RuntimeWarning)
/var/folders/9n/zs24nr4n1jq9k_zjkcrvfj1h0000gn/T/ipykernel_9108/2584104940.py:2: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, context_recall
/var/folders/9n/zs24nr4n1jq9k_zjkcrvfj1h0000gn/T/ipykernel_9108/2584104940.py:2: DeprecationWarning: Importing context_recall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_recall
  from ragas.metrics import faithfulness, context_recall


In [22]:
single_pred = all_predictions[0]

single_eval_input = {
    'question': single_pred['question'],
    'answer': single_pred['answer'],
    'contexts': single_pred['contexts'],
    'ground_truth': single_pred['ground_truth']
}

faith_score = faithfulness_chain(single_eval_input)
recall_score = context_recall_chain(single_eval_input)

print(f'Faithfulness score:   {faith_score["faithfulness"]:.3f}')
print(f'Context Recall score: {recall_score["context_recall"]:.3f}')
print()
print('Uitleg:')
print('  Faithfulness = 1.0  → model verzint niets, alles staat in de bronnen')
print('  Context Recall = 1.0 → de opgehaalde bronnen bevatten alle info uit de ground truth')

/var/folders/9n/zs24nr4n1jq9k_zjkcrvfj1h0000gn/T/ipykernel_9108/1939107940.py:10: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use invoke instead.
  faith_score = faithfulness_chain(single_eval_input)


Faithfulness score:   0.500
Context Recall score: 1.000

Uitleg:
  Faithfulness = 1.0  → model verzint niets, alles staat in de bronnen
  Context Recall = 1.0 → de opgehaalde bronnen bevatten alle info uit de ground truth


### Sectie 2: Test met foute antwoorden (lage scores verwacht)

In [23]:
# Test 1: Fout antwoord → lage faithfulness
fake_faith_input = single_eval_input.copy()
fake_faith_input['answer'] = 'New York heeft een bevolking van 100 miljoen en ligt op Mars.'

fake_faith = faithfulness_chain(fake_faith_input)
print('--- Test met fout antwoord ---')
print(f'Originele faithfulness: {faith_score["faithfulness"]:.3f}')
print(f'Neppe faithfulness:     {fake_faith["faithfulness"]:.3f}')
print('→ Lager, want het antwoord staat NIET in de bronnen\n')

# Test 2: Irrelevante bronnen → lage context recall
fake_context_input = single_eval_input.copy()
fake_context_input['contexts'] = [
    'Ik hou van pizza en ijs.',
    'Het weer is vandaag mooi.'
]

fake_recall = context_recall_chain(fake_context_input)
print('--- Test met irrelevante bronnen ---')
print(f'Originele context recall: {recall_score["context_recall"]:.3f}')
print(f'Neppe context recall:     {fake_recall["context_recall"]:.3f}')
print('→ Lager, want de bronnen bevatten NIET de info uit de ground truth')

--- Test met fout antwoord ---
Originele faithfulness: 0.500
Neppe faithfulness:     0.000
→ Lager, want het antwoord staat NIET in de bronnen

--- Test met irrelevante bronnen ---
Originele context recall: 1.000
Neppe context recall:     1.000
→ Lager, want de bronnen bevatten NIET de info uit de ground truth


### Sectie 3: Batch evaluatie (LangChain stijl)

In [24]:
eval_inputs = [
    {
        'question': p['question'],
        'answer': p['answer'],
        'contexts': p['contexts'],
        'ground_truth': p['ground_truth'],
    }
    for p in all_predictions
]

faith_batch = faithfulness_chain.batch(eval_inputs)
recall_batch = context_recall_chain.batch(eval_inputs)

faith_scores = [r['faithfulness'] for r in faith_batch]
recall_scores = [r['context_recall'] for r in recall_batch]

print('=== Batch RAGAS Resultaten ===')
print(f'Gemiddelde Faithfulness:   {sum(faith_scores)/len(faith_scores):.3f}')
print(f'Gemiddelde Context Recall: {sum(recall_scores)/len(recall_scores):.3f}')
print()
for i, (f, r) in enumerate(zip(faith_scores, recall_scores)):
    print(f'  Vraag {i+1}: faithfulness={f:.2f}, context_recall={r:.2f}')

=== Batch RAGAS Resultaten ===
Gemiddelde Faithfulness:   0.900
Gemiddelde Context Recall: 0.800

  Vraag 1: faithfulness=1.00, context_recall=1.00
  Vraag 2: faithfulness=0.50, context_recall=1.00
  Vraag 3: faithfulness=1.00, context_recall=1.00
  Vraag 4: faithfulness=1.00, context_recall=0.00
  Vraag 5: faithfulness=1.00, context_recall=1.00


### Sectie 4: RAGAS Native evaluatie (productie-methode)

In [26]:
from ragas import evaluate
from datasets import Dataset

dataset_all = Dataset.from_list([
    {
        'question': p['question'],
        'answer': p['answer'],
        'contexts': p['contexts'],
        'reference': p['ground_truth'],  # nieuwere RAGAS gebruikt 'reference'
    }
    for p in all_predictions
])

print('=== RAGAS Native Evaluatie ===')

print('\n--- Faithfulness ---')
f_result = evaluate(dataset_all, metrics=[faithfulness])
print(f_result)

print('\n--- Context Recall ---')
r_result = evaluate(dataset_all, metrics=[context_recall])
print(r_result)

print('\n--- Gecombineerde metrics ---')
combined = evaluate(dataset_all, metrics=[faithfulness, context_recall])
print(combined)

=== RAGAS Native Evaluatie ===

--- Faithfulness ---


Evaluating:   0%|          | 0/5 [00:00<?, ?it/s]

{'faithfulness': 0.8333}

--- Context Recall ---


Evaluating:   0%|          | 0/5 [00:00<?, ?it/s]

{'context_recall': 0.8000}

--- Gecombineerde metrics ---


Evaluating:   0%|          | 0/10 [00:00<?, ?it/s]

{'faithfulness': 0.7333, 'context_recall': 0.8000}


---
# Samenvatting: Wat hebben we geleerd?

| Concept | Wat is het? |
|---|---|
| **RAG** | Systeem dat documenten ophaalt en dan een LLM gebruikt om te antwoorden |
| **Vectorstore** | Database die tekst opslaat als wiskundige vectoren voor snelle zoekacties |
| **Handmatige evaluatie** | Jij leest de output en beoordeelt zelf |
| **LLM-as-a-Judge** | Een LLM beoordeelt de output van een andere LLM |
| **Faithfulness** | Verzint het model dingen? (1.0 = nooit) |
| **Context Recall** | Worden de juiste bronnen opgehaald? (1.0 = altijd) |
| **RAGAS** | Framework voor systematische RAG-evaluatie |
